## Ricavo funzione
Fit dei dati della misura con solo sorgente + primo polarizzatore.
Se la sorgente è parzialmente polarizzata: I1(θ) = (I0/2)[1 + p cos(2θ − 2φ0)], cioè periodo 180° (2θ), non 4θ.

In [7]:
import ROOT
import numpy as np
import pandas as pd

ROOT.gStyle.SetOptFit(1111)      # box con chi2/ndf, probabilità e parametri (metti 0 per toglierlo)
ROOT.gStyle.SetFitFormat("4.2e")

### 1. Dati

In [8]:
FILE = "DatiSorgenteSim.xlsx"
FOGLIO = "Polarizzata"
N_SCARTA = 6                    # quanti punti iniziali scartare (0 = nessuno)

df = pd.read_excel(FILE, sheet_name=FOGLIO)
df = df.dropna(subset=["θ [rad]", "V", "sigma_θ", "sigma_V"])
df = df.iloc[N_SCARTA:]

x     = df["θ [rad]"].to_numpy(dtype=float)
y     = df["V"].to_numpy(dtype=float)
err_x = df["sigma_θ"].to_numpy(dtype=float)
err_y = df["sigma_V"].to_numpy(dtype=float)

### 2. Fit

In [9]:
# Modello: V(θ) = A * (1 + E*cos(2θ - F)) * (1 + D*θ)
#   A = livello medio,  E = grado di polarizzazione P,  F = 2*φ0 (fase),  D = deriva lineare
modello = "[0]*(1 + [1]*cos(2*x - [2]))*(1 + [3]*x)"

fit_f = ROOT.TF1("fit_f", modello, x.min(), x.max())
E0 = (y.max() - y.min())/(y.max() + y.min())                  # valore di partenza di E
fit_f.SetParameters(y.mean(), E0, 2*x[np.argmax(y)], 0)       # F parte da 2*(posizione del massimo)
fit_f.SetParNames("A", "E", "F", "D")

grafico = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)    
grafico.SetTitle("Dati sperimentali e fit; #theta [rad]; V [V]")
grafico.SetMarkerStyle(20)
grafico.SetMarkerSize(0.7)
grafico.SetMarkerColor(ROOT.kAzure + 2)
grafico.SetLineColor(ROOT.kAzure + 3)

fit_res = grafico.Fit(fit_f, "S")

print(f"Fit status: {fit_res.Status()}")
for i, nome in enumerate(["A", "E", "F", "D"]):
    print(f"{nome} = {fit_f.GetParameter(i)} ± {fit_f.GetParError(i)}")
print(f"Chi2 = {fit_res.Chi2()}, ndf = {fit_res.Ndf()}")

# (E, F) e (-E, F+π) descrivono la stessa curva scelgo E positivo e F tra 0 e 2π
E_new, errE_new = fit_f.GetParameter(1), fit_f.GetParError(1)
F_new, errF_new = fit_f.GetParameter(2), fit_f.GetParError(2)
if E_new < 0: # qui controllo che E sia positivo altrimenti visto che il coseno è pari, cambio segno a E e aggiungo π a F
    E_new, F_new = -E_new, F_new + np.pi
F_new = F_new % (2*np.pi)

print(f"E = {E_new*100:.2f} ± {errE_new*100:.2f} %   (contrasto, limite inferiore di p)")
print(f"F = {F_new:.3f} ± {errF_new:.3f} rad = {np.degrees(F_new):.1f}°  ->  phi0 = F/2 = {np.degrees(F_new)/2:.1f}°")

if E_new < 3*errE_new:
    print("E compatibile con 0: sorgente non polarizzata")
else:
    print(f"E NON compatibile con 0 ({E_new/errE_new:.1f} sigma): sorgente parzialmente polarizzata")

Fit status: 0****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      111.572
NDf                       =          190
Edm                       =  4.05932e-06
NCalls                    =           97
A                         =      3.50007   +/-   0.00186502  
E                         =    0.0602841   +/-   0.000368419 
F                         =      2.72335   +/-   0.00576283  
D                         =  -0.00109269   +/-   0.000132304 

A = 3.5000656124199154 ± 0.0018650230269365601
E = 0.060284146519917856 ± 0.00036841941094669224
F = 2.723354072198801 ± 0.005762834128066999
D = -0.0010926908158386268 ± 0.00013230408522330015
Chi2 = 111.57183952122904, ndf = 190
E = 6.03 ± 0.04 %   (contrasto, limite inferiore di p)
F = 2.723 ± 0.006 rad = 156.0°  ->  phi0 = F/2 = 78.0°
E NON compatibile con 0 (163.6 sigma): sorgente parzialmente polarizzata


In [10]:
from scipy.stats import f as f_dist

g0 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
fit_0 = ROOT.TF1("fit_0", "[0]*(1 + [1]*x)", x.min(), x.max())
fit_0.SetParameters(y.mean(), 0)
res_0 = g0.Fit(fit_0, "SQ0")

f_stat = ((res_0.Chi2() - fit_res.Chi2()) / (res_0.Ndf() - fit_res.Ndf())) / (fit_res.Chi2() / fit_res.Ndf())
print(f"F = {f_stat:.1f},  p = {f_dist.sf(f_stat, res_0.Ndf() - fit_res.Ndf(), fit_res.Ndf()):.2e}")

F = 23713.9,  p = 1.24e-228


### 3. Grafico

In [11]:
c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()
grafico.Draw("AP")

leg = ROOT.TLegend(0.14, 0.14, 0.42, 0.23)
leg.AddEntry(grafico, "Dati", "pe")
leg.AddEntry(fit_f, "Fit", "l")
leg.Draw()

c1.Draw()
c1.SaveAs("Immagini/sorgente_fit.png")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file Immagini/sorgente_fit.png has been created


## Confronto
Confronto E ed F ricavati ora (solo sorgente + P1) con quelli ricavati dalla misura completa di Malus, dove la sorgente veniva descritta dal fattore (1 + E cos(2θ - F)). Se la causa è la polarizzazione parziale della sorgente i due risultati devono essere compatibili.

Il confronto di F ha senso solo se la scala del polarizzatore (lo zero) e la posizione/orientazione della lampada non sono cambiate rispetto alla misura di prima.

In [12]:
# valori ottenuti dal fit della misura di Malus
E_old, errE_old = 0.0673, 0.0024
F_old, errF_old = 2.777, 0.036            # rad

#  confronto di E 
diff_E = E_new - E_old                            # di quanto differiscono
err_diff_E = np.sqrt(errE_new**2 + errE_old**2)       # incertezza della differenza
sigma_E = diff_E / err_diff_E                            # differenza in unità di sigma

#  confronto di F  
diff_F = F_new - F_old

# riporto la differenza tra -π e +π
if diff_F > np.pi:
    diff_F = diff_F - 2*np.pi
if diff_F < -np.pi:
    diff_F = diff_F + 2*np.pi

err_diff_F = np.sqrt(errF_new**2 + errF_old**2)
sigma_F = diff_F / err_diff_F

print(f"E: ora = {E_new*100:.2f} ± {errE_new*100:.2f} %   prima = {E_old*100:.2f} ± {errE_old*100:.2f} %   differenza = {sigma_E:.1f} sigma")
print(f"F: ora = {F_new:.3f} ± {errF_new:.3f} rad   prima = {F_old:.3f} ± {errF_old:.3f} rad   differenza = {sigma_F:.1f} sigma")

if abs(sigma_E) < 3 and abs(sigma_F) < 3:
    print("Compatibili entro 3 sigma: la polarizzazione della sorgente spiega l'onda trovata.")

E: ora = 6.03 ± 0.04 %   prima = 6.73 ± 0.24 %   differenza = -2.9 sigma
F: ora = 2.723 ± 0.006 rad   prima = 2.777 ± 0.036 rad   differenza = -1.5 sigma
Compatibili entro 3 sigma: la polarizzazione della sorgente spiega l'onda trovata.
